# Makemore Lesson 02

Live notes and code while following Andrej Karpathy's makemore Part 2.

## Session Goals

- Follow the lesson actively.
- Keep code runnable.
- Add short notes only when a concept is confusing or important.


In [21]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline


## Data


In [22]:
words = open('../data/names.txt', 'r').read().splitlines()
words[:8], len(words)


(['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia'],
 32033)

## Notes

- 


In [ ]:
# stoi, itos

In [23]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [24]:
block_size = 3 # context length: how many characters do we take to predict the next one?
X, Y = [], []
for w in words[:5]:
  
  #print(w)
  context = [0] * block_size
  for ch in w + '.':
    ix = stoi[ch]
    X.append(context)
    Y.append(ix)
    print(''.join(itos[i] for i in context), '--->', itos[ix])
    context = context[1:] + [ix] # crop and append
  
X = torch.tensor(X)
Y = torch.tensor(Y)

... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .
... ---> a
..a ---> v
.av ---> a
ava ---> .
... ---> i
..i ---> s
.is ---> a
isa ---> b
sab ---> e
abe ---> l
bel ---> l
ell ---> a
lla ---> .
... ---> s
..s ---> o
.so ---> p
sop ---> h
oph ---> i
phi ---> a
hia ---> .


In [27]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

In [13]:
C = torch.randn((27, 2)) #Embedding means representing discrete tokens as continuous vectors.

C

tensor([[ 0.2401, -0.5400],
        [-1.1058, -1.0842],
        [ 1.2614,  0.7596],
        [-0.7423, -1.3111],
        [-0.4029,  1.4708],
        [ 0.9855,  1.0697],
        [ 1.7095,  0.8275],
        [ 0.0625,  1.2957],
        [-1.0032,  0.3002],
        [ 0.8014, -0.1413],
        [-0.7550, -0.1045],
        [-2.0373, -0.1113],
        [ 0.0604,  0.6716],
        [-2.4536, -0.2455],
        [-0.6730,  0.8873],
        [ 0.8212,  0.6360],
        [ 0.4380,  2.8910],
        [-2.3392, -0.8821],
        [ 0.6182,  0.1859],
        [-0.1402,  1.5047],
        [ 0.0687, -0.0404],
        [ 2.0149,  1.6668],
        [ 0.0938,  0.3313],
        [-0.1175,  0.1495],
        [-0.6222,  1.8209],
        [ 1.5911, -0.4088],
        [ 0.6632,  0.5942]])

In [14]:
C[5]

tensor([0.9855, 1.0697])

In [17]:
F.one_hot(torch.tensor(5), num_classes=27).float() @ C

tensor([0.9855, 1.0697])

In [28]:
C[X]

tensor([[[ 0.2401, -0.5400],
         [ 0.2401, -0.5400],
         [ 0.2401, -0.5400]],

        [[ 0.2401, -0.5400],
         [ 0.2401, -0.5400],
         [ 0.9855,  1.0697]],

        [[ 0.2401, -0.5400],
         [ 0.9855,  1.0697],
         [-2.4536, -0.2455]],

        [[ 0.9855,  1.0697],
         [-2.4536, -0.2455],
         [-2.4536, -0.2455]],

        [[-2.4536, -0.2455],
         [-2.4536, -0.2455],
         [-1.1058, -1.0842]],

        [[ 0.2401, -0.5400],
         [ 0.2401, -0.5400],
         [ 0.2401, -0.5400]],

        [[ 0.2401, -0.5400],
         [ 0.2401, -0.5400],
         [ 0.8212,  0.6360]],

        [[ 0.2401, -0.5400],
         [ 0.8212,  0.6360],
         [ 0.0604,  0.6716]],

        [[ 0.8212,  0.6360],
         [ 0.0604,  0.6716],
         [ 0.8014, -0.1413]],

        [[ 0.0604,  0.6716],
         [ 0.8014, -0.1413],
         [ 0.0938,  0.3313]],

        [[ 0.8014, -0.1413],
         [ 0.0938,  0.3313],
         [ 0.8014, -0.1413]],

        [[ 0.0938,  0

In [73]:
emb = C[X]
emb.shape

torch.Size([32, 3, 10])

In [74]:
W1 = torch.randn((6, 100))
b1 = torch.randn(100)

In [75]:
emb @ W1 + b1 # make emb.shape into [32, 6]

RuntimeError: mat1 and mat2 shapes cannot be multiplied (96x10 and 6x100)

In [76]:
torch.cat([emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]], 1).shape

torch.Size([32, 30])

In [77]:
torch.cat(torch.unbind(emb, 1), 1).shape

torch.Size([32, 30])

In [78]:
emb.view(32, 6)

RuntimeError: shape '[32, 6]' is invalid for input of size 960

In [70]:
h = torch.tanh(emb.view(-1, 6) @ W1 + b1) # broadcasting

RuntimeError: mat1 and mat2 shapes cannot be multiplied (160x6 and 30x200)

In [71]:
h

tensor([[-0.6323,  0.0997,  0.5766,  ...,  0.8827,  0.9157,  0.6947],
        [-0.9222, -0.7665, -0.6345,  ..., -0.5993,  0.9993,  0.9999],
        [-0.7243,  0.4586,  0.9959,  ...,  0.7964,  0.9859, -0.9977],
        ...,
        [-0.9999,  0.9637,  0.9830,  ...,  0.7961, -0.3976, -0.9813],
        [ 0.3455,  0.9991,  0.9777,  ...,  0.6518, -0.9963,  0.9416],
        [ 0.9992, -0.5394,  0.9135,  ...,  0.6168, -0.3008, -0.2494]])

In [51]:
W2 = torch.randn((100, 27))
b2 = torch.randn(27)

In [53]:
logits = h @ W2 + b2
logits

tensor([[ 2.2481e+00,  1.4140e+00,  1.0190e+01, -2.1688e+00, -1.5032e+01,
         -1.0847e+01,  8.5508e+00,  7.2052e+00,  5.6489e+00, -2.6964e+00,
          8.8579e+00,  1.2941e+00, -2.9302e+00,  2.0911e+00, -5.2167e+00,
          2.0205e+00,  1.3848e+01,  3.8618e+00, -8.6924e+00,  1.0546e+01,
         -3.2035e-01, -1.0785e+01, -2.9228e-03, -4.9020e+00,  6.7768e+00,
          1.0169e+01,  4.8376e+00],
        [ 5.9160e+00,  4.3860e+00,  3.0447e+00,  8.2896e+00, -1.9437e+01,
         -9.2739e+00,  7.4596e+00,  1.5535e+01, -2.1810e+00,  9.5246e+00,
          1.1361e+01,  7.4568e+00,  4.7111e+00, -1.9909e+00, -1.4243e+00,
         -1.3993e+01, -1.5282e+01,  6.9501e+00,  2.2496e+00,  1.2508e+01,
         -1.8977e+00,  3.5606e+00, -1.6162e+01,  5.8136e+00,  9.1366e-01,
          2.8360e+00,  5.0675e+00],
        [-1.8615e+00, -9.5391e+00,  1.2084e+00, -1.4849e+01, -9.8663e+00,
         -7.4454e+00, -3.6509e-01,  5.6241e+00, -2.7869e-01,  8.8187e+00,
          7.5854e+00, -2.2570e+00, -1.48

In [54]:
counts = logits.exp()
prob = counts / counts.sum(1, keepdim=True)

In [55]:
prob[0].sum()

tensor(1.0000)

In [57]:
loss = -prob[torch.arange(32), Y].log().mean() 
loss

#prob: 32 个样本，每个样本对 27 个字符的预测概率
#Y: 32 个样本，每个样本真正的下一个字符 index

tensor(18.9255)

In [82]:
g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((27, 2), generator=g)
W1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)
W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [83]:
X.shape, Y.shape

(torch.Size([32, 3]), torch.Size([32]))

In [84]:
emb = C[X]
h = torch.tanh(emb.view(-1, 6) @ W1 + b1) # broadcasting
logits = h @ W2 + b2
# counts = logits.exp()
# prob = counts / counts.sum(1, keepdim=True)
# loss = -prob[torch.arange(32), Y].log().mean()
loss = F.cross_entropy(logits, Y)
#  backward & forward pass can be more efficient
loss

tensor(17.7697)

In [85]:
for p in parameters:
    p.requires_grad = True

In [116]:

for _ in range(10):
    emb = C[X]
    h = torch.tanh(emb.view(-1, 6) @ W1 + b1) # broadcasting
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Y)
    print(loss.item())
    
    for p in parameters:
        p.grad = None
    
    loss.backward()
    
    for p in parameters:
        p.data += -0.1 * p.grad

0.26385238766670227
0.26381486654281616
0.26377755403518677
0.2637404799461365
0.2637036442756653
0.2636669874191284
0.26363053917884827
0.2635943591594696
0.26355835795402527
0.26352259516716003


In [117]:
logits.max(0)

TypeError: 'builtin_function_or_method' object is not subscriptable